To access GPU from the notebook, follow the following steps:
- `idev -N 1 -c 10 --gres=gpu:1 -t 360`
- activate your cuda environment
- `srun jupyter notebook --no-browser --port=8888 --ip=0.0.0.0`
- connect to existing jupyter server as the kernel


In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="gpt2-grpo-imdb",
    learning_rate=1e-5,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,          # How many completions to generate per prompt
    max_completion_length=128,
    logging_steps=10,
    # report_to="wandb"
)

# Load dataset and filter short reviews
dataset = load_dataset("mteb/imdb", split="train")
dataset = dataset.filter(lambda x: len(x["text"]) > 200, batched=False)

In [16]:
dataset[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [17]:
def generate_prompts(example):
    # Take the first 50 characters as the starting prompt for GPT-2
    return {"prompt": example["text"][:50]}

dataset = dataset.map(generate_prompts)
dataset[0]

Map: 100%|██████████| 24800/24800 [00:00<00:00, 28846.83 examples/s]


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
# Load the sentiment classifier
sentiment_pipe = pipeline(
    "sentiment-analysis", 
    model="lvwerra/distilbert-imdb", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

def sentiment_reward_func(completions, **kwargs):
    # 'completions' are the string outputs generated by GPT-2
    rewards = []
    results = sentiment_pipe(completions)
    
    for res in results:
        # We want to maximize positive sentiment
        if res["label"] == "POSITIVE":
            rewards.append(res["score"])
        else:
            rewards.append(-res["score"]) # Penalize negative sentiment
            
    return rewards

In [ ]:
model_id = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Initialize the trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[sentiment_reward_func],
    args=training_args,
    train_dataset=dataset,
)

# Run the training loop automatically
trainer.train()

In [18]:
dataset[0]['prompt']

'I rented I AM CURIOUS-YELLOW from my video store b'

In [ ]:
save_path = "./grpo_positive"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


In [19]:
generator = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. Provide a neutral or slightly negative prompt to see if the model forces a positive spin
test_prompt = "I went to the theater with low expectations, and honestly"

# Generate the completion
outputs = generator(
    # test_prompt, 
    dataset[0]['prompt'],
    max_new_tokens=50, 
    pad_token_id=tokenizer.eos_token_id
)
generated_text = outputs[0]["generated_text"]

print("--- Generated Review ---")
print(generated_text)

# 4. Verify the actual sentiment score of the generated output
sentiment_pipe = pipeline(
    "sentiment-analysis", 
    model="lvwerra/distilbert-imdb", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

score = sentiment_pipe(generated_text)
print("\n--- Sentiment Assessment ---")
print(score)

--- Generated Review ---
I rented I AM CURIOUS-YELLOW from my video store bib and it is a great piece and perfect piece and captures the film it captures it works it beautifully and captures it perfectly and captures it captures it perfect work it beautifully and captures it beautifully each and every time it works it makes it perfect and perfect pieces


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 29718.46it/s]



--- Sentiment Assessment ---
[{'label': 'POSITIVE', 'score': 0.9967249035835266}]
